# Fixlane Concern Parser — Evaluation Notebook

This notebook compares three models on the held-out test set:
1. **Fine-tuned Qwen2.5-3B** — our LoRA adapter from training
2. **Base Qwen2.5-3B (no fine-tuning)** — same model, zero-shot, no adapter
3. **Claude Sonnet (few-shot prompted)** — the strong frontier baseline the assignment asks for

The base model comparison shows how much fine-tuning helped.  
The Claude Sonnet comparison shows how we stack up against a frontier model.

**You need:**
- `adapter_weights.zip` from training
- `test.jsonl`
- An Anthropic API key (free credits at console.anthropic.com — 100 calls costs under $0.50)
- T4 GPU runtime in Colab

**If you don't have an API key yet:** Run Steps 1–12 (fine-tune vs base model). Skip Step 13. You can come back and run Step 13 later when you have the key.

## Step 1 — Install dependencies

In [ ]:
!pip install -q transformers peft bitsandbytes rouge-score tqdm anthropic

## Step 2 — Upload files

In [ ]:
from google.colab import files
import os
import zipfile

print("Upload test.jsonl and adapter_weights.zip")
uploaded = files.upload()

# Unzip the adapter weights if they came as a zip
if "adapter_weights.zip" in uploaded:
    os.makedirs("/content/adapter_weights", exist_ok=True)
    with zipfile.ZipFile("/content/adapter_weights.zip", "r") as zf:
        zf.extractall("/content/adapter_weights")
    print("Adapter weights extracted!")
    print("Files:", os.listdir("/content/adapter_weights"))

TEST_PATH    = "/content/test.jsonl"
ADAPTER_PATH = "/content/adapter_weights"
OUTPUT_PATH  = "/content/eval_results.json"

print("\nPaths set!")

## Step 3 — Paste your Anthropic API key

Get a free key at https://console.anthropic.com — sign up, go to API Keys, create one.  
100 test calls costs under $0.50 with the free credits they give you.  
If you don't have one yet, just leave this as-is and skip Step 13.

In [ ]:
# Paste your Anthropic API key here
ANTHROPIC_API_KEY = "sk-ant-..."   # replace with your actual key

# This flag controls whether the Claude Sonnet baseline runs.
# It gets set to True automatically in Step 13 when you run that cell.
RUN_CLAUDE_BASELINE = False

## Step 4 — Imports

In [ ]:
import json
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from rouge_score import rouge_scorer
import anthropic

print("Imports done!")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Step 5 — Load the test data

In [ ]:
def load_jsonl(file_path):
    rows = []
    with open(file_path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

test_data = load_jsonl(TEST_PATH)
print(f"Loaded {len(test_data)} test examples")
print()
print("First example:")
print("  Input:", test_data[0]["input"])
print("  Expected:", json.dumps(test_data[0]["output"], indent=4))

## Step 6 — Prompt builder

Same instruction template as training — must match word for word.

In [ ]:
# This must be identical to the instruction in train.ipynb.
# The fine-tuned model was trained to respond to this exact format.
# We also use it as the base for the Claude Sonnet prompt so the
# comparison is as fair as possible.
INSTRUCTION = """You are a vehicle repair assistant. Parse the technician note below into structured fields.

Respond with a valid JSON object containing exactly these fields:
- vehicle_system: one of [engine, electrical, brakes, suspension, hvac, drivetrain, body, other]
- primary_symptom: a short noun phrase describing the symptom
- severity: one of [safety_critical, drivability, comfort, cosmetic]
- suggested_diagnostic: a short imperative phrase, or null

Only output the JSON object. No extra text."""


def build_prompt(technician_note):
    return INSTRUCTION + "\n\nTechnician note: " + technician_note + "\n\nJSON:"


print("Prompt builder ready!")

## Step 7 — Load quantization config

Same 4-bit setup for both local models so the comparison is fair.

In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Quantization config ready!")

## Step 8 — Load the fine-tuned model (base + LoRA adapter)

In [ ]:
print("Loading tokenizer from adapter checkpoint...")
ft_tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, trust_remote_code=True)
if ft_tokenizer.pad_token is None:
    ft_tokenizer.pad_token = ft_tokenizer.eos_token
ft_tokenizer.padding_side = "right"

print("Loading base model in 4-bit...")
ft_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True,
)

print("Attaching LoRA adapter...")
ft_model = PeftModel.from_pretrained(ft_base, ADAPTER_PATH)
ft_model.eval()

print("Fine-tuned model ready!")

## Step 9 — Load the base model (no adapter)

Same Qwen2.5-3B but without our LoRA adapter. This shows the fine-tuning delta.

In [ ]:
print("Loading base tokenizer...")
base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if base_tokenizer.pad_token is None:
    base_tokenizer.pad_token = base_tokenizer.eos_token
base_tokenizer.padding_side = "right"

print("Loading base model in 4-bit (no adapter)...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True,
)
base_model.eval()

print("Base model ready!")

## Step 10 — Local model inference helper

In [ ]:
def run_local_inference(model, tokenizer, technician_note):
    """
    Runs a local model on one technician note.
    Works for both the fine-tuned model and the base model.
    Returns the raw string output.
    """
    prompt = build_prompt(technician_note)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,   # greedy — deterministic and easy to reproduce
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    # Decode only the new tokens — skip the input prompt
    prompt_len = inputs["input_ids"].shape[1]
    new_tokens = output_ids[0][prompt_len:]
    raw_output = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return raw_output.strip()


# Quick sanity check
test_note = test_data[0]["input"]
print("Test note:", test_note)
print()
print("Fine-tuned output:", run_local_inference(ft_model, ft_tokenizer, test_note))
print()
print("Base model output:", run_local_inference(base_model, base_tokenizer, test_note))

## Step 11 — Output parsing and scoring

In [ ]:
def parse_output(raw_output):
    clean = raw_output.strip()
    # Strip markdown backticks if present
    if clean.startswith("```"):
        lines = clean.split("\n")
        lines = [l for l in lines if not l.strip().startswith("```")]
        clean = "\n".join(lines).strip()
    
    # Extract just the first { ... } block and ignore anything after it.
    # The model sometimes keeps generating text after the JSON — this handles that.
    start = clean.find('{')
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(clean)):
        if clean[i] == '{':
            depth += 1
        elif clean[i] == '}':
            depth -= 1
            if depth == 0:
                try:
                    return json.loads(clean[start:i+1])
                except json.JSONDecodeError:
                    return None
    return None

def score_prediction(predicted_dict, expected_dict):
    """
    Score one prediction against the expected output.
    Returns a dict with a score for each field.
    """
    # If output couldn't be parsed as JSON, everything scores 0
    if predicted_dict is None:
        return {
            "json_valid":            0,
            "vehicle_system_exact":  0,
            "severity_exact":        0,
            "primary_symptom_rouge": 0.0,
            "diagnostic_rouge":      0.0,
        }

    scores = {}
    scores["json_valid"] = 1

    # vehicle_system and severity are fixed label sets — exact match is right
    pred_system = predicted_dict.get("vehicle_system", "")
    scores["vehicle_system_exact"] = 1 if pred_system == expected_dict["vehicle_system"] else 0

    pred_sev = predicted_dict.get("severity", "")
    scores["severity_exact"] = 1 if pred_sev == expected_dict["severity"] else 0

    # primary_symptom and suggested_diagnostic are free text — ROUGE-L measures
    # word overlap without penalizing valid rephrasing
    rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

    pred_symptom     = predicted_dict.get("primary_symptom", "") or ""
    expected_symptom = expected_dict.get("primary_symptom", "") or ""
    scores["primary_symptom_rouge"] = rouge.score(expected_symptom, pred_symptom)["rougeL"].fmeasure

    pred_diag     = predicted_dict.get("suggested_diagnostic", "") or ""
    expected_diag = expected_dict.get("suggested_diagnostic", "") or ""
    scores["diagnostic_rouge"] = rouge.score(expected_diag, pred_diag)["rougeL"].fmeasure

    return scores


print("Scoring functions ready!")

## Step 12 — Run fine-tune and base model on all 100 test examples

Takes about 5–10 minutes on a T4.

In [ ]:
finetuned_results = []
base_results      = []

print(f"Evaluating fine-tuned and base models on {len(test_data)} test examples...")
print()

for i, row in enumerate(test_data):
    note     = row["input"]
    expected = row["output"]

    if (i + 1) % 10 == 0:
        print(f"  {i + 1}/{len(test_data)} done...")

    # Fine-tuned model
    ft_raw    = run_local_inference(ft_model, ft_tokenizer, note)
    ft_parsed = parse_output(ft_raw)
    ft_scores = score_prediction(ft_parsed, expected)
    finetuned_results.append({
        "input": note, "expected": expected,
        "raw": ft_raw, "parsed": ft_parsed, "scores": ft_scores,
    })

    # Base model (no adapter)
    base_raw    = run_local_inference(base_model, base_tokenizer, note)
    base_parsed = parse_output(base_raw)
    base_scores = score_prediction(base_parsed, expected)
    base_results.append({
        "input": note, "expected": expected,
        "raw": base_raw, "parsed": base_parsed, "scores": base_scores,
    })

print("\nDone with local models!")

## Step 13 — Claude Sonnet baseline (requires API key)

This is the frontier model comparison the assignment asks for.  
We give Claude Sonnet the same task with 3 few-shot examples from the training set.  

**If you don't have an API key yet, skip this cell and go to Step 14.**  
The results table in Step 14 will just show dashes for the Claude column.

In [ ]:
# These 3 few-shot examples are from the training set.
# They cover different vehicle systems and severity levels so Claude
# understands the full range of what we're asking for.
FEW_SHOT_EXAMPLES = [
    {
        "input": "Loud squealnig from front brks when stopping",
        "output": {
            "vehicle_system": "brakes",
            "primary_symptom": "front brake squealing under braking",
            "severity": "safety_critical",
            "suggested_diagnostic": "inspect front brake pads and rotors for wear"
        }
    },
    {
        "input": "heater not getting warm in cold weather. needs attention",
        "output": {
            "vehicle_system": "hvac",
            "primary_symptom": "heater no heat",
            "severity": "comfort",
            "suggested_diagnostic": "test heater operation and check coolant level"
        }
    },
    {
        "input": "Note: per customer, turn signal stays on after turning, doesn't cancel",
        "output": {
            "vehicle_system": "electrical",
            "primary_symptom": "turn signal cancel failure",
            "severity": "drivability",
            "suggested_diagnostic": "inspect turn signal stalk and cancel mechanism"
        }
    },
]


def build_claude_system_prompt():
    """
    Build the system prompt + few-shot examples for Claude Sonnet.
    We include the label definitions and 3 real training examples
    so Claude understands the exact taxonomy we're using.
    """
    system_msg = (
        "You are a vehicle repair assistant. Parse technician notes into structured JSON.\n\n"
        "Always respond with a valid JSON object with exactly these fields:\n"
        "- vehicle_system: one of [engine, electrical, brakes, suspension, hvac, drivetrain, body, other]\n"
        "- primary_symptom: a short noun phrase describing the symptom\n"
        "- severity: one of [safety_critical, drivability, comfort, cosmetic]\n"
        "  Definitions: safety_critical = affects ability to safely operate the vehicle; "
        "drivability = affects normal use but not safety; comfort = affects convenience or pleasure; "
        "cosmetic = appearance only\n"
        "- suggested_diagnostic: a short imperative phrase, or null\n\n"
        "Only output the JSON. No explanation, no markdown, no extra text.\n\n"
        "Examples:\n"
    )
    for ex in FEW_SHOT_EXAMPLES:
        system_msg += f"\nInput: {ex['input']}\n"
        system_msg += f"Output: {json.dumps(ex['output'])}\n"
    return system_msg


def run_claude_inference(client, technician_note):
    """
    Runs Claude Sonnet on one technician note.
    Returns the raw string output.
    """
    system_prompt = build_claude_system_prompt()

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=256,
        system=system_prompt,
        messages=[
            {"role": "user", "content": f"Input: {technician_note}\nOutput:"}
        ],
    )

    return response.content[0].text.strip()


# --- Run Claude Sonnet on all 100 test examples ---
claude_results = []

# Make sure the API key is set before we start
if ANTHROPIC_API_KEY.startswith("sk-ant-..."):
    print("No API key set — skipping Claude Sonnet baseline.")
    print("Paste your key in Step 3 and re-run this cell when ready.")
else:
    RUN_CLAUDE_BASELINE = True
    claude_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

    print(f"Running Claude Sonnet on {len(test_data)} test examples...")
    print("(Each call takes ~1-2 seconds — total about 2-3 minutes)")
    print()

    for i, row in enumerate(test_data):
        note     = row["input"]
        expected = row["output"]

        if (i + 1) % 10 == 0:
            print(f"  {i + 1}/{len(test_data)} done...")

        claude_raw    = run_claude_inference(claude_client, note)
        claude_parsed = parse_output(claude_raw)
        claude_scores = score_prediction(claude_parsed, expected)

        claude_results.append({
            "input": note, "expected": expected,
            "raw": claude_raw, "parsed": claude_parsed, "scores": claude_scores,
        })

    print("\nClaude Sonnet baseline done!")

## Step 14 — Print results table

In [ ]:
def average_scores(results_list):
    """
    Average per-example scores across all test examples.
    """
    n = len(results_list)
    total_json    = 0
    total_system  = 0
    total_sev     = 0
    total_symptom = 0.0
    total_diag    = 0.0

    for r in results_list:
        s = r["scores"]
        total_json    += s["json_valid"]
        total_system  += s["vehicle_system_exact"]
        total_sev     += s["severity_exact"]
        total_symptom += s["primary_symptom_rouge"]
        total_diag    += s["diagnostic_rouge"]

    return {
        "json_valid_pct":          round(total_json / n * 100, 1),
        "vehicle_system_accuracy": round(total_system / n * 100, 1),
        "severity_accuracy":       round(total_sev / n * 100, 1),
        "primary_symptom_rougeL":  round(total_symptom / n, 3),
        "diagnostic_rougeL":       round(total_diag / n, 3),
    }


ft_avg   = average_scores(finetuned_results)
base_avg = average_scores(base_results)

# Claude column only if we ran it
if RUN_CLAUDE_BASELINE:
    claude_avg = average_scores(claude_results)
    claude_col = claude_avg
else:
    claude_col = None


def fmt(val):
    # Format a value for the table, or show a dash if not available
    if val is None:
        return "—"
    return str(val)


print("=" * 75)
print("RESULTS TABLE")
print("=" * 75)
print(f"{'Metric':<35} {'Fine-tuned':>12} {'Base model':>12} {'Claude Sonnet':>13}")
print("-" * 75)

metrics = [
    ("JSON valid (%)",              "json_valid_pct"),
    ("vehicle_system accuracy (%)", "vehicle_system_accuracy"),
    ("severity accuracy (%)",       "severity_accuracy"),
    ("primary_symptom ROUGE-L",     "primary_symptom_rougeL"),
    ("suggested_diagnostic ROUGE-L","diagnostic_rougeL"),
]

for label, key in metrics:
    ft_val   = fmt(ft_avg[key])
    base_val = fmt(base_avg[key])
    cl_val   = fmt(claude_col[key] if claude_col else None)
    print(f"{label:<35} {ft_val:>12} {base_val:>12} {cl_val:>13}")

print("=" * 75)

if not RUN_CLAUDE_BASELINE:
    print("\nNote: Claude Sonnet column is empty — paste your API key in Step 3 and re-run Step 13.")

## Step 15 — Look at failure examples

In [ ]:
# Cases where the fine-tune got vehicle_system wrong
print("--- Fine-tune: vehicle_system failures ---")
fail_count = 0
for r in finetuned_results:
    if r["scores"]["vehicle_system_exact"] == 0:
        fail_count += 1
        if fail_count <= 5:
            pred = r["parsed"] or {}
            print(f"  Input:     {r['input']}")
            print(f"  Expected:  {r['expected']['vehicle_system']}")
            print(f"  Predicted: {pred.get('vehicle_system', 'N/A')}")
            print()
print(f"Total: {fail_count} failures")

In [ ]:
# Cases where the fine-tune got severity wrong
print("--- Fine-tune: severity failures ---")
sev_fail = 0
for r in finetuned_results:
    if r["scores"]["severity_exact"] == 0:
        sev_fail += 1
        if sev_fail <= 5:
            pred = r["parsed"] or {}
            print(f"  Input:     {r['input']}")
            print(f"  Expected:  {r['expected']['severity']}")
            print(f"  Predicted: {pred.get('severity', 'N/A')}")
            print()
print(f"Total: {sev_fail} failures")

In [ ]:
# Cases where fine-tune won but base model lost — shows the fine-tuning made a real difference
print("--- Cases fine-tune won, base model lost (vehicle_system) ---")
win_count = 0
for ft_r, base_r in zip(finetuned_results, base_results):
    ft_correct   = ft_r["scores"]["vehicle_system_exact"] == 1
    base_correct = base_r["scores"]["vehicle_system_exact"] == 1
    if ft_correct and not base_correct:
        win_count += 1
        if win_count <= 5:
            ft_pred   = ft_r["parsed"] or {}
            base_pred = base_r["parsed"] or {}
            print(f"  Input:          {ft_r['input']}")
            print(f"  Expected:       {ft_r['expected']['vehicle_system']}")
            print(f"  Fine-tune said: {ft_pred.get('vehicle_system', 'N/A')}  <- correct")
            print(f"  Base said:      {base_pred.get('vehicle_system', 'N/A')}  <- wrong")
            print()
print(f"Total wins over base model: {win_count}")

## Step 16 — Save results and download

In [ ]:
# Build the output dict — include Claude results only if we ran them
output_data = {
    "finetuned_aggregate":  ft_avg,
    "base_model_aggregate": base_avg,
    "per_example_finetuned": finetuned_results,
    "per_example_base":      base_results,
}

if RUN_CLAUDE_BASELINE:
    output_data["claude_sonnet_aggregate"] = claude_avg
    output_data["per_example_claude"]      = claude_results

with open(OUTPUT_PATH, "w") as f:
    json.dump(output_data, f, indent=2)

print(f"Results saved to {OUTPUT_PATH}")

from google.colab import files
files.download(OUTPUT_PATH)
print("Download started!")